In [304]:
using LowLevelFEM, LinearAlgebra

In [305]:
using SparseArrays #timing

"""
    reductionMatrices(P::Problem)

Construct sparse transformation matrices for reducing the polynomial order
of a C0 Lagrange finite element field from order `p` to `p - 1`.

The returned matrices satisfy

    u_full = T * u_reduced
    u_reduced = R * u_full

for fields representable in the reduced space.

The transformation is constructed from the Gmsh Lagrange basis functions.
All element types belonging to the problem must have the same polynomial
order. An error is thrown for first-order meshes.

The matrices are expanded automatically according to `P.pdim`.
"""
function reductionMatrices(P::Problem)

    gmsh.model.setCurrent(P.name)

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    function element_family(name::String)
        if occursin("Line", name)
            return "Line"
        elseif occursin("Triangle", name)
            return "Triangle"
        elseif occursin("Quadrilateral", name) ||
               occursin("Quadrangle", name)
            return "Quadrangle"
        elseif occursin("Tetrahedron", name)
            return "Tetrahedron"
        elseif occursin("Hexahedron", name)
            return "Hexahedron"
        elseif occursin("Prism", name)
            return "Prism"
        elseif occursin("Pyramid", name)
            return "Pyramid"
        else
            error("reductionMatrices: unsupported Gmsh element family \"$name\".")
        end
    end

    function local_to_3d(localCoord, dim, n)
        ξ = zeros(Float64, 3 * n)

        @inbounds for a in 1:n
            for d in 1:dim
                ξ[3*(a-1)+d] =
                    localCoord[dim*(a-1)+d]
            end
        end

        return ξ
    end

    # ------------------------------------------------------------------
    # 1. Node coordinates
    # ------------------------------------------------------------------

    println("\n[1] Node coordinates")
    @time begin

        nodeTags, coord, _ =
            gmsh.model.mesh.getNodes(-1, -1, false, false)

        nodeCoord = Dict{UInt64,NTuple{3,Float64}}()
        sizehint!(nodeCoord, length(nodeTags))

        @inbounds for i in eachindex(nodeTags)
            nodeCoord[nodeTags[i]] = (
                coord[3*i-2],
                coord[3*i-1],
                coord[3*i]
            )
        end
    end

    # ------------------------------------------------------------------
    # 2. Collect elements + build element-type cache
    # ------------------------------------------------------------------

    println("\n[2] Collect elements + local transformation cache")
    @time begin

        elements = NamedTuple[]
        seen_elements = Set{UInt64}()
        orders = Set{Int}()

        type_cache = Dict{Int,Any}()

        for mat in P.material

            dimTags =
                gmsh.model.getEntitiesForPhysicalName(mat.phName)

            for (edim, etag) in dimTags

                edim == P.dim || continue

                elemTypes, elemTags, elemNodeTags =
                    gmsh.model.mesh.getElements(edim, etag)

                for it in eachindex(elemTypes)

                    et = elemTypes[it]

                    name, dim, p, nHigh, ξHigh0, nPrimary =
                        gmsh.model.mesh.getElementProperties(et)

                    push!(orders, p)

                    if !haskey(type_cache, et)

                        p > 1 ||
                            error(
                                "reductionMatrices: polynomial order must be greater than one."
                            )

                        q = p - 1
                        family = element_family(name)

                        etLow =
                            gmsh.model.mesh.getElementType(
                                family,
                                q,
                                false
                            )

                        nameLow, dimLow, qCheck,
                        nLow, ξLow0, nPrimaryLow =
                            gmsh.model.mesh.getElementProperties(etLow)

                        dimLow == dim ||
                            error(
                                "reductionMatrices: incompatible reduced element dimension."
                            )

                        qCheck == q ||
                            error(
                                "reductionMatrices: could not construct order-$q element."
                            )

                        ξHigh = local_to_3d(ξHigh0, dim, nHigh)

                        _, funT, _ =
                            gmsh.model.mesh.getBasisFunctions(
                                etLow,
                                ξHigh,
                                "Lagrange"
                            )

                        Te =
                            Matrix(
                                transpose(
                                    reshape(funT, nLow, nHigh)
                                )
                            )

                        ξLow = local_to_3d(ξLow0, dim, nLow)

                        _, funR, _ =
                            gmsh.model.mesh.getBasisFunctions(
                                et,
                                ξLow,
                                "Lagrange"
                            )

                        Re =
                            Matrix(
                                transpose(
                                    reshape(funR, nHigh, nLow)
                                )
                            )

                        type_cache[et] = (
                            etLow=etLow,
                            p=p,
                            q=q,
                            nHigh=nHigh,
                            nLow=nLow,
                            nPrimary=nPrimary,
                            nPrimaryLow=nPrimaryLow,
                            Te=Te,
                            Re=Re
                        )
                    end

                    cache = type_cache[et]

                    tags = elemTags[it]
                    conn = elemNodeTags[it]

                    @inbounds for e in eachindex(tags)

                        elemTag = tags[e]

                        elemTag in seen_elements && continue
                        push!(seen_elements, elemTag)

                        o = (e - 1) * cache.nHigh

                        nodes =
                            UInt64.(
                                conn[(o+1):(o+cache.nHigh)]
                            )

                        push!(
                            elements,
                            (
                                tag=elemTag,
                                et=et,
                                nodes=nodes,
                                primary=nodes[1:cache.nPrimary]
                            )
                        )
                    end
                end
            end
        end
    end

    isempty(elements) &&
        error("reductionMatrices: no domain elements found.")

    length(orders) == 1 ||
        error(
            "reductionMatrices: the mesh must have homogeneous polynomial order; found $(sort!(collect(orders)))."
        )

    p = first(orders)

    p > 1 ||
        error(
            "reductionMatrices: polynomial order must be greater than one."
        )

    # ------------------------------------------------------------------
    # 3. Characteristic length
    # ------------------------------------------------------------------

    println("\n[3] Characteristic length")
    @time begin

        used_nodes = Set{UInt64}()

        for elem in elements
            union!(used_nodes, elem.nodes)
        end

        xmin = Inf
        ymin = Inf
        zmin = Inf
        xmax = -Inf
        ymax = -Inf
        zmax = -Inf

        for node in used_nodes
            x, y, z = nodeCoord[node]

            xmin = min(xmin, x)
            ymin = min(ymin, y)
            zmin = min(zmin, z)

            xmax = max(xmax, x)
            ymax = max(ymax, y)
            zmax = max(zmax, z)
        end

        L = max(xmax - xmin, ymax - ymin, zmax - zmin)

        L > 0 ||
            error("reductionMatrices: degenerate mesh geometry.")

        tol = 1e-10 * L
        tol2 = tol^2
    end

    # ------------------------------------------------------------------
    # 4. Reduced global node numbering
    # ------------------------------------------------------------------

    println("\n[4] Reduced global node numbering")
    @time begin

        nElem = length(elements)

        reduced_conn =
            Vector{Vector{Int}}(undef, nElem)

        reduced_coord =
            Vector{Matrix{Float64}}(undef, nElem)

        vertex_to_reduced =
            Dict{UInt64,Int}()

        node_to_elements =
            Dict{UInt64,Vector{Int}}()

        nReduced = 0

        for ie in 1:nElem

            elem = elements[ie]
            cache = type_cache[elem.et]

            nHigh = cache.nHigh
            nLow = cache.nLow

            Xhigh = Matrix{Float64}(undef, 3, nHigh)
            #Xhigh = zeros(Float64, 3, nHigh)

            @inbounds for a in 1:nHigh
                x, y, z = nodeCoord[elem.nodes[a]]

                Xhigh[1, a] = x
                Xhigh[2, a] = y
                Xhigh[3, a] = z
            end

            Xlow =
                transpose(cache.Re * transpose(Xhigh))

            reduced_coord[ie] = Xlow

            #rconn = zeros(Int, nLow)
            rconn = Vector{Int}(undef, nLow)

            neighbour_count = Dict{Int,Int}()

            for node in elem.primary
                for je in get(node_to_elements, node, Int[])
                    neighbour_count[je] =
                        get(neighbour_count, je, 0) + 1
                end
            end

            neighbours =
                [
                    je for (je, count) in neighbour_count
                           if count >= 2
                ]

            @inbounds for a in 1:nLow

                if a <= cache.nPrimaryLow

                    original_vertex = elem.nodes[a]

                    if haskey(vertex_to_reduced, original_vertex)

                        rconn[a] =
                            vertex_to_reduced[original_vertex]

                    else

                        nReduced += 1

                        vertex_to_reduced[original_vertex] =
                            nReduced

                        rconn[a] = nReduced
                    end

                    continue
                end

                xa = Xlow[1, a]
                ya = Xlow[2, a]
                za = Xlow[3, a]

                found = 0

                for je in neighbours

                    Xold = reduced_coord[je]
                    rold = reduced_conn[je]

                    for b in axes(Xold, 2)

                        dx = xa - Xold[1, b]
                        dy = ya - Xold[2, b]
                        dz = za - Xold[3, b]

                        if dx^2 + dy^2 + dz^2 <= tol2
                            found = rold[b]
                            break
                        end
                    end

                    found != 0 && break
                end

                if found == 0
                    nReduced += 1
                    rconn[a] = nReduced
                else
                    rconn[a] = found
                end
            end

            reduced_conn[ie] = rconn

            for node in elem.primary
                push!(
                    get!(node_to_elements, node, Int[]),
                    ie
                )
            end
        end
    end

    # ------------------------------------------------------------------
    # 5. Assemble scalar T
    # ------------------------------------------------------------------

    println("\n[5] Assemble Tn")
    @time begin

        IT = Int[]
        JT = Int[]
        VT = Float64[]
        sizehint!(IT, P.non * maximum(c.nLow for c in values(type_cache)))
        sizehint!(JT, length(IT))
        sizehint!(VT, length(IT))

        full_seen = falses(P.non)

        for ie in 1:nElem

            elem = elements[ie]
            cache = type_cache[elem.et]
            rconn = reduced_conn[ie]

            Te = cache.Te

            @inbounds for a in 1:cache.nHigh

                node = Int(elem.nodes[a])

                node <= P.non ||
                    error(
                        "reductionMatrices: node tag $node exceeds problem.non=$(P.non)."
                    )

                full_seen[node] && continue
                full_seen[node] = true

                for b in 1:cache.nLow

                    v = Te[a, b]

                    abs(v) < 100eps(Float64) && continue

                    push!(IT, node)
                    push!(JT, rconn[b])
                    push!(VT, v)
                end
            end
        end

        Tn =
            sparse(
                IT,
                JT,
                VT,
                P.non,
                nReduced
            )
    end

    # ------------------------------------------------------------------
    # 6. Assemble scalar R
    # ------------------------------------------------------------------

    println("\n[6] Assemble Rn")
    @time begin

        IR = Int[]
        JR = Int[]
        VR = Float64[]

        reduced_seen = falses(nReduced)

        for ie in 1:nElem

            elem = elements[ie]
            cache = type_cache[elem.et]
            rconn = reduced_conn[ie]

            Re = cache.Re

            @inbounds for a in 1:cache.nLow

                rnode = rconn[a]

                reduced_seen[rnode] && continue
                reduced_seen[rnode] = true

                for b in 1:cache.nHigh

                    v = Re[a, b]

                    abs(v) < 100eps(Float64) && continue

                    push!(IR, rnode)
                    push!(JR, Int(elem.nodes[b]))
                    push!(VR, v)
                end
            end
        end

        Rn =
            sparse(
                IR,
                JR,
                VR,
                nReduced,
                P.non
            )
    end

    # ------------------------------------------------------------------
    # 7. Expand according to pdim
    # ------------------------------------------------------------------

    println("\n[7] Expand according to pdim")
    @time begin

        d = P.pdim

        Id = spdiagm(0 => ones(Float64, d))

        T = kron(Tn, Id)
        R = kron(Rn, Id)
    end

    return T, R
end

reductionMatrices

In [306]:
using SparseArrays

"""
    reductionMatrices(P::Problem)

Construct sparse transformation matrices for reducing the polynomial order
of a C0 Lagrange finite element field from order `p` to `p - 1`.

The returned matrices satisfy

    u_full = T * u_reduced
    u_reduced = R * u_full

for fields representable in the reduced space.

The transformation is constructed from the Gmsh Lagrange basis functions.
All element types belonging to the problem must have the same polynomial
order. An error is thrown for first-order meshes.

The matrices are expanded automatically according to `P.pdim`.
"""
function reductionMatrices(P::Problem)

    gmsh.model.setCurrent(P.name)

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    function element_family(name::String)
        if occursin("Line", name)
            return "Line"
        elseif occursin("Triangle", name)
            return "Triangle"
        elseif occursin("Quadrilateral", name) ||
               occursin("Quadrangle", name)
            return "Quadrangle"
        elseif occursin("Tetrahedron", name)
            return "Tetrahedron"
        elseif occursin("Hexahedron", name)
            return "Hexahedron"
        elseif occursin("Prism", name)
            return "Prism"
        elseif occursin("Pyramid", name)
            return "Pyramid"
        else
            error("reductionMatrices: unsupported Gmsh element family \"$name\".")
        end
    end

    # getElementProperties returns dim coordinates/node,
    # while getBasisFunctions expects (u,v,w) triplets.
    function local_to_3d(localCoord, dim, n)
        ξ = zeros(Float64, 3*n)

        @inbounds for a in 1:n
            for d in 1:dim
                ξ[3*(a-1)+d] =
                    localCoord[dim*(a-1)+d]
            end
        end

        return ξ
    end

    # ------------------------------------------------------------------
    # Node coordinates
    # ------------------------------------------------------------------

    nodeTags, coord, _ =
        gmsh.model.mesh.getNodes(-1, -1, false, false)

    nodeCoord = Dict{UInt64,NTuple{3,Float64}}()
    sizehint!(nodeCoord, length(nodeTags))

    @inbounds for i in eachindex(nodeTags)
        nodeCoord[nodeTags[i]] = (
            coord[3*i-2],
            coord[3*i-1],
            coord[3*i]
        )
    end

    # ------------------------------------------------------------------
    # Collect all elements belonging to the Problem
    # ------------------------------------------------------------------

    elements = NamedTuple[]
    seen_elements = Set{UInt64}()
    orders = Set{Int}()

    # Cache data depending only on element type
    type_cache = Dict{Int,Any}()

    for mat in P.material

        dimTags =
            gmsh.model.getEntitiesForPhysicalName(mat.phName)

        for (edim, etag) in dimTags

            # Only domain elements are relevant here
            edim == P.dim || continue

            elemTypes, elemTags, elemNodeTags =
                gmsh.model.mesh.getElements(edim, etag)

            for it in eachindex(elemTypes)

                et = elemTypes[it]

                name, dim, p, nHigh, ξHigh0, nPrimary =
                    gmsh.model.mesh.getElementProperties(et)

                push!(orders, p)

                if !haskey(type_cache, et)

                    p > 1 ||
                        error(
                            "reductionMatrices: polynomial order must be greater than one."
                        )

                    q = p - 1
                    family = element_family(name)

                    etLow =
                        gmsh.model.mesh.getElementType(
                            family,
                            q,
                            false
                        )

                    nameLow, dimLow, qCheck,
                    nLow, ξLow0, nPrimaryLow =
                        gmsh.model.mesh.getElementProperties(etLow)

                    dimLow == dim ||
                        error(
                            "reductionMatrices: incompatible reduced element dimension."
                        )

                    qCheck == q ||
                        error(
                            "reductionMatrices: could not construct order-$q element."
                        )

                    # ----------------------------------------------
                    # Local prolongation:
                    #
                    #   u_high = Te * u_low
                    #
                    # Evaluate low-order basis at high-order nodes.
                    # ----------------------------------------------

                    ξHigh = local_to_3d(ξHigh0, dim, nHigh)

                    _, funT, _ =
                        gmsh.model.mesh.getBasisFunctions(
                            etLow,
                            ξHigh,
                            "Lagrange"
                        )

                    Te =
                        Matrix(
                            transpose(
                                reshape(funT, nLow, nHigh)
                            )
                        )

                    # ----------------------------------------------
                    # Local restriction:
                    #
                    #   u_low = Re * u_high
                    #
                    # Evaluate high-order basis at low-order nodes.
                    # ----------------------------------------------

                    ξLow = local_to_3d(ξLow0, dim, nLow)

                    _, funR, _ =
                        gmsh.model.mesh.getBasisFunctions(
                            et,
                            ξLow,
                            "Lagrange"
                        )

                    Re =
                        Matrix(
                            transpose(
                                reshape(funR, nHigh, nLow)
                            )
                        )

                    type_cache[et] = (
                        etLow=etLow,
                        p=p,
                        q=q,
                        nHigh=nHigh,
                        nLow=nLow,
                        nPrimary=nPrimary,
                        nPrimaryLow=nPrimaryLow,
                        Te=Te,
                        Re=Re
                    )
                end

                cache = type_cache[et]

                tags = elemTags[it]
                conn = elemNodeTags[it]

                @inbounds for e in eachindex(tags)

                    elemTag = tags[e]

                    elemTag in seen_elements && continue
                    push!(seen_elements, elemTag)

                    o = (e - 1) * cache.nHigh

                    nodes =
                        UInt64.(
                            conn[(o+1):(o+cache.nHigh)]
                        )

                    push!(
                        elements,
                        (
                            tag=elemTag,
                            et=et,
                            nodes=nodes,
                            primary=nodes[1:cache.nPrimary]
                        )
                    )
                end
            end
        end
    end

    isempty(elements) &&
        error("reductionMatrices: no domain elements found.")

    length(orders) == 1 ||
        error(
            "reductionMatrices: the mesh must have homogeneous polynomial order; found $(sort!(collect(orders)))."
        )

    p = first(orders)

    p > 1 ||
        error(
            "reductionMatrices: polynomial order must be greater than one."
        )

    # ------------------------------------------------------------------
    # Characteristic length for coordinate comparisons
    # ------------------------------------------------------------------

    used_nodes = Set{UInt64}()

    for elem in elements
        union!(used_nodes, elem.nodes)
    end

    xmin = Inf
    ymin = Inf
    zmin = Inf
    xmax = -Inf
    ymax = -Inf
    zmax = -Inf

    for node in used_nodes
        x, y, z = nodeCoord[node]

        xmin = min(xmin, x)
        ymin = min(ymin, y)
        zmin = min(zmin, z)

        xmax = max(xmax, x)
        ymax = max(ymax, y)
        zmax = max(zmax, z)
    end

    L = max(xmax - xmin, ymax - ymin, zmax - zmin)

    L > 0 ||
        error("reductionMatrices: degenerate mesh geometry.")

    tol = 1e-10 * L
    tol2 = tol^2

    # ------------------------------------------------------------------
    # Build reduced global node numbering
    # ------------------------------------------------------------------

    nElem = length(elements)

    reduced_conn =
        Vector{Vector{Int}}(undef, nElem)

    reduced_coord =
        Vector{Matrix{Float64}}(undef, nElem)

    # Original vertex node -> reduced node
    vertex_to_reduced =
        Dict{UInt64,Int}()

    # Original primary node -> already processed elements
    node_to_elements =
        Dict{UInt64,Vector{Int}}()

    nReduced = 0

    for ie in 1:nElem

        elem = elements[ie]
        cache = type_cache[elem.et]

        nHigh = cache.nHigh
        nLow = cache.nLow

        # ----------------------------------------------
        # Physical coordinates of high-order nodes
        # ----------------------------------------------

        Xhigh = zeros(Float64, 3, nHigh)

        @inbounds for a in 1:nHigh
            x, y, z = nodeCoord[elem.nodes[a]]

            Xhigh[1, a] = x
            Xhigh[2, a] = y
            Xhigh[3, a] = z
        end

        # Physical positions of reduced nodes:
        #
        # Xlow = Re * Xhigh'
        #
        Xlow =
            transpose(cache.Re * transpose(Xhigh))

        reduced_coord[ie] = Xlow

        rconn = zeros(Int, nLow)

        # ----------------------------------------------
        # Potential already-processed neighbours.
        #
        # Count shared primary nodes.
        # ----------------------------------------------

        neighbour_count = Dict{Int,Int}()

        for node in elem.primary
            for je in get(node_to_elements, node, Int[])
                neighbour_count[je] =
                    get(neighbour_count, je, 0) + 1
            end
        end

        neighbours =
            [
                je for (je, count) in neighbour_count
                       if count >= 2
            ]

        # ----------------------------------------------
        # Reduced nodes
        # ----------------------------------------------

        @inbounds for a in 1:nLow

            # Vertices are guaranteed to correspond to the original
            # primary nodes and can be identified exactly by node tag.
            if a <= cache.nPrimaryLow

                original_vertex = elem.nodes[a]

                if haskey(vertex_to_reduced, original_vertex)

                    rconn[a] =
                        vertex_to_reduced[original_vertex]

                else

                    nReduced += 1

                    vertex_to_reduced[original_vertex] =
                        nReduced

                    rconn[a] = nReduced
                end

                continue
            end

            # Non-vertex reduced node: search only in topologically
            # connected previously processed elements.
            xa = Xlow[1, a]
            ya = Xlow[2, a]
            za = Xlow[3, a]

            found = 0

            for je in neighbours

                Xold = reduced_coord[je]
                rold = reduced_conn[je]

                for b in axes(Xold, 2)

                    dx = xa - Xold[1, b]
                    dy = ya - Xold[2, b]
                    dz = za - Xold[3, b]

                    if dx^2 + dy^2 + dz^2 <= tol2
                        found = rold[b]
                        break
                    end
                end

                found != 0 && break
            end

            if found == 0
                nReduced += 1
                rconn[a] = nReduced
            else
                rconn[a] = found
            end
        end

        reduced_conn[ie] = rconn

        # Register this element as processed
        for node in elem.primary
            push!(
                get!(node_to_elements, node, Int[]),
                ie
            )
        end
    end

    # ------------------------------------------------------------------
    # Assemble scalar T
    # ------------------------------------------------------------------

    IT = Int[]
    JT = Int[]
    VT = Float64[]

    full_seen = falses(P.non)

    for ie in 1:nElem

        elem = elements[ie]
        cache = type_cache[elem.et]
        rconn = reduced_conn[ie]

        Te = cache.Te

        @inbounds for a in 1:cache.nHigh

            node = Int(elem.nodes[a])

            node <= P.non ||
                error(
                    "reductionMatrices: node tag $node exceeds problem.non=$(P.non)."
                )

            full_seen[node] && continue
            full_seen[node] = true

            for b in 1:cache.nLow

                v = Te[a, b]

                abs(v) < 100eps(Float64) && continue

                push!(IT, node)
                push!(JT, rconn[b])
                push!(VT, v)
            end
        end
    end

    Tn =
        sparse(
            IT,
            JT,
            VT,
            P.non,
            nReduced
        )

    # ------------------------------------------------------------------
    # Assemble scalar R
    #
    # A reduced node can belong to several elements. Its interpolation
    # row is taken from the first processed element containing it.
    # ------------------------------------------------------------------

    IR = Int[]
    JR = Int[]
    VR = Float64[]

    reduced_seen = falses(nReduced)

    for ie in 1:nElem

        elem = elements[ie]
        cache = type_cache[elem.et]
        rconn = reduced_conn[ie]

        Re = cache.Re

        @inbounds for a in 1:cache.nLow

            rnode = rconn[a]

            reduced_seen[rnode] && continue
            reduced_seen[rnode] = true

            for b in 1:cache.nHigh

                v = Re[a, b]

                abs(v) < 100eps(Float64) && continue

                push!(IR, rnode)
                push!(JR, Int(elem.nodes[b]))
                push!(VR, v)
            end
        end
    end

    Rn =
        sparse(
            IR,
            JR,
            VR,
            nReduced,
            P.non
        )

    # ------------------------------------------------------------------
    # Expand according to pdim
    #
    # DOF ordering:
    #
    # node1_comp1, node1_comp2, ...,
    # node2_comp1, ...
    # ------------------------------------------------------------------

    d = P.pdim

    Id = spdiagm(0 => ones(Float64, d))

    T = kron(Tn, Id)
    R = kron(Rn, Id)

    return T, R
end

reductionMatrices

In [307]:
using SparseArrays                # optimalizált
using LinearAlgebra

"""
    reductionMatrices(P::Problem)

Construct sparse transformation matrices for reducing the polynomial order
of a C0 Lagrange finite element field from order `p` to `p - 1`.

The returned matrices satisfy

    u_full = T * u_reduced
    u_reduced = R * u_full

for fields representable in the reduced space.

The transformation is constructed from the Gmsh Lagrange basis functions.
All element types belonging to the problem must have the same polynomial
order. An error is thrown for first-order meshes.

The matrices are expanded automatically according to `P.pdim`.
"""
function reductionMatrices(P::Problem)

    gmsh.model.setCurrent(P.name)

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------

    function element_family(name::String)
        if occursin("Line", name)
            return "Line"
        elseif occursin("Triangle", name)
            return "Triangle"
        elseif occursin("Quadrilateral", name) ||
               occursin("Quadrangle", name)
            return "Quadrangle"
        elseif occursin("Tetrahedron", name)
            return "Tetrahedron"
        elseif occursin("Hexahedron", name)
            return "Hexahedron"
        elseif occursin("Prism", name)
            return "Prism"
        elseif occursin("Pyramid", name)
            return "Pyramid"
        else
            error(
                "reductionMatrices: unsupported Gmsh element family \"$name\"."
            )
        end
    end

    function local_to_3d(localCoord, dim, n)
        ξ = zeros(Float64, 3 * n)

        @inbounds for a in 1:n
            o3 = 3 * (a - 1)
            od = dim * (a - 1)

            for d in 1:dim
                ξ[o3+d] = localCoord[od+d]
            end
        end

        return ξ
    end

    # ------------------------------------------------------------------
    # Node coordinates
    #
    # LLFEM already uses node tags directly for global DOF numbering,
    # therefore the same assumption is used here: node tags are in
    # 1:P.non.
    # ------------------------------------------------------------------

    nodeTags, coord, _ =
        gmsh.model.mesh.getNodes(-1, -1, false, false)

    nodeCoord = Matrix{Float64}(undef, 3, P.non)

    @inbounds for i in eachindex(nodeTags)

        node = Int(nodeTags[i])

        1 <= node <= P.non ||
            error(
                "reductionMatrices: node tag $node is incompatible with P.non=$(P.non)."
            )

        nodeCoord[1, node] = coord[3*i-2]
        nodeCoord[2, node] = coord[3*i-1]
        nodeCoord[3, node] = coord[3*i]
    end

    # ------------------------------------------------------------------
    # Element-type data
    #
    # Parallel concretely typed arrays are used instead of Dict{Int,Any}.
    # Each element stores only the integer index of its element type.
    # ------------------------------------------------------------------

    type_id = Dict{Int,Int}()

    nHighs = Int[]
    nLows = Int[]
    nPrimarys = Int[]
    nPrimaryLows = Int[]

    Tes = Matrix{Float64}[]
    Res = Matrix{Float64}[]

    # Reusable workspaces for physical node coordinates
    Xhigh_work = Matrix{Float64}[]
    Xlow_work = Matrix{Float64}[]

    # ------------------------------------------------------------------
    # Elements belonging to the Problem
    # ------------------------------------------------------------------

    elem_type_id = Int[]
    elem_nodes = Vector{UInt64}[]

    seen_elements = Set{UInt64}()

    p_global = 0

    xmin = Inf
    ymin = Inf
    zmin = Inf

    xmax = -Inf
    ymax = -Inf
    zmax = -Inf

    for mat in P.material

        dimTags =
            gmsh.model.getEntitiesForPhysicalName(mat.phName)

        for (edim, etag) in dimTags

            edim == P.dim || continue

            elemTypes, elemTags, elemNodeTags =
                gmsh.model.mesh.getElements(edim, etag)

            for it in eachindex(elemTypes)

                et = Int(elemTypes[it])

                # ------------------------------------------------------
                # Initialize element-type cache once
                # ------------------------------------------------------

                tid = get(type_id, et, 0)

                if tid == 0

                    name, dim, p, nHigh, ξHigh0, nPrimary =
                        gmsh.model.mesh.getElementProperties(et)

                    p > 1 ||
                        error(
                            "reductionMatrices: polynomial order must be greater than one."
                        )

                    if p_global == 0
                        p_global = p
                    elseif p != p_global
                        error(
                            "reductionMatrices: the mesh must have homogeneous polynomial order; " *
                            "found both order $p_global and order $p."
                        )
                    end

                    q = p - 1

                    family = element_family(name)

                    etLow =
                        gmsh.model.mesh.getElementType(
                            family,
                            q,
                            false
                        )

                    _, dimLow, qCheck,
                    nLow, ξLow0, nPrimaryLow =
                        gmsh.model.mesh.getElementProperties(etLow)

                    dimLow == dim ||
                        error(
                            "reductionMatrices: incompatible reduced element dimension."
                        )

                    qCheck == q ||
                        error(
                            "reductionMatrices: could not construct order-$q element."
                        )

                    # --------------------------------------------------
                    # Local prolongation matrix
                    #
                    # u_high = Te * u_low
                    # --------------------------------------------------

                    ξHigh =
                        local_to_3d(
                            ξHigh0,
                            dim,
                            nHigh
                        )

                    _, funT, _ =
                        gmsh.model.mesh.getBasisFunctions(
                            etLow,
                            ξHigh,
                            "Lagrange"
                        )

                    Te =
                        Matrix(
                            transpose(
                                reshape(
                                    funT,
                                    nLow,
                                    nHigh
                                )
                            )
                        )

                    # --------------------------------------------------
                    # Local restriction matrix
                    #
                    # u_low = Re * u_high
                    # --------------------------------------------------

                    ξLow =
                        local_to_3d(
                            ξLow0,
                            dim,
                            nLow
                        )

                    _, funR, _ =
                        gmsh.model.mesh.getBasisFunctions(
                            et,
                            ξLow,
                            "Lagrange"
                        )

                    Re =
                        Matrix(
                            transpose(
                                reshape(
                                    funR,
                                    nHigh,
                                    nLow
                                )
                            )
                        )

                    push!(nHighs, nHigh)
                    push!(nLows, nLow)
                    push!(nPrimarys, nPrimary)
                    push!(nPrimaryLows, nPrimaryLow)

                    push!(Tes, Te)
                    push!(Res, Re)

                    # Store coordinates as node × xyz:
                    #
                    # Xlow = Re * Xhigh
                    #
                    push!(
                        Xhigh_work,
                        Matrix{Float64}(undef, nHigh, 3)
                    )

                    push!(
                        Xlow_work,
                        Matrix{Float64}(undef, nLow, 3)
                    )

                    tid = length(nHighs)
                    type_id[et] = tid

                else

                    # Homogeneous-order check also for already known types
                    _, _, p, _, _, _ =
                        gmsh.model.mesh.getElementProperties(et)

                    p == p_global ||
                        error(
                            "reductionMatrices: the mesh must have homogeneous polynomial order."
                        )
                end

                # ------------------------------------------------------
                # Collect connectivity
                # ------------------------------------------------------

                nHigh = nHighs[tid]

                tags = elemTags[it]
                conn = elemNodeTags[it]

                @inbounds for e in eachindex(tags)

                    elemTag = tags[e]

                    elemTag in seen_elements && continue
                    push!(seen_elements, elemTag)

                    o = (e - 1) * nHigh

                    nodes = Vector{UInt64}(undef, nHigh)

                    copyto!(
                        nodes,
                        1,
                        conn,
                        o + 1,
                        nHigh
                    )

                    push!(elem_type_id, tid)
                    push!(elem_nodes, nodes)

                    # Bounding box at no additional mesh traversal cost
                    for nodeTag in nodes

                        node = Int(nodeTag)

                        x = nodeCoord[1, node]
                        y = nodeCoord[2, node]
                        z = nodeCoord[3, node]

                        xmin = min(xmin, x)
                        ymin = min(ymin, y)
                        zmin = min(zmin, z)

                        xmax = max(xmax, x)
                        ymax = max(ymax, y)
                        zmax = max(zmax, z)
                    end
                end
            end
        end
    end

    isempty(elem_nodes) &&
        error(
            "reductionMatrices: no domain elements found."
        )

    p_global > 1 ||
        error(
            "reductionMatrices: polynomial order must be greater than one."
        )

    # ------------------------------------------------------------------
    # Coordinate comparison tolerance
    # ------------------------------------------------------------------

    L =
        max(
            xmax - xmin,
            ymax - ymin,
            zmax - zmin
        )

    L > 0 ||
        error(
            "reductionMatrices: degenerate mesh geometry."
        )

    tol2 = (1e-10 * L)^2

    # ------------------------------------------------------------------
    # Global reduced-node numbering
    # ------------------------------------------------------------------

    nElem = length(elem_nodes)

    reduced_conn =
        Vector{Vector{Int}}(undef, nElem)

    # Global physical coordinates of reduced nodes.
    # A coordinate is stored only once per global reduced node.
    xr = Float64[]
    yr = Float64[]
    zr = Float64[]

    # Original primary node -> reduced node.
    # Direct indexing is possible because LLFEM already relies on
    # contiguous Gmsh node tags.
    vertex_to_reduced =
        zeros(Int, P.non)

    # Primary node -> already processed elements touching that node.
    #
    # Vectors are allocated lazily only for primary nodes actually used.
    node_to_elements =
        Vector{Union{Nothing,Vector{Int}}}(undef, P.non)

    fill!(node_to_elements, nothing)

    # Workspace for neighbour detection without allocating a Dict
    # for every element.
    neighbour_hits = zeros(Int, nElem)

    touched = Int[]
    sizehint!(touched, 32)

    nReduced = 0

    for ie in 1:nElem

        tid = elem_type_id[ie]

        nodes = elem_nodes[ie]

        nHigh = nHighs[tid]
        nLow = nLows[tid]
        nPrimary = nPrimarys[tid]
        nPrimaryLow = nPrimaryLows[tid]

        Re = Res[tid]

        Xhigh = Xhigh_work[tid]
        Xlow = Xlow_work[tid]

        # --------------------------------------------------------------
        # Physical coordinates of the high-order element nodes
        # --------------------------------------------------------------

        @inbounds for a in 1:nHigh

            node = Int(nodes[a])

            Xhigh[a, 1] = nodeCoord[1, node]
            Xhigh[a, 2] = nodeCoord[2, node]
            Xhigh[a, 3] = nodeCoord[3, node]
        end

        # --------------------------------------------------------------
        # Physical coordinates of reduced nodes
        #
        # Xlow = Re * Xhigh
        #
        # No temporary matrix is allocated.
        # --------------------------------------------------------------

        mul!(
            Xlow,
            Re,
            Xhigh
        )

        rconn = Vector{Int}(undef, nLow)

        # --------------------------------------------------------------
        # Find previously processed topological neighbours.
        #
        # A shared non-vertex C0 node can only lie on a common edge/face.
        # Such elements share at least two primary nodes.
        # --------------------------------------------------------------

        empty!(touched)

        @inbounds for a in 1:nPrimary

            node = Int(nodes[a])

            lst = node_to_elements[node]

            lst === nothing && continue

            for je in lst

                if neighbour_hits[je] == 0
                    push!(touched, je)
                end

                neighbour_hits[je] += 1
            end
        end

        # --------------------------------------------------------------
        # Number reduced nodes
        # --------------------------------------------------------------

        @inbounds for a in 1:nLow

            # ----------------------------------------------------------
            # Vertex nodes can be identified exactly by original
            # Gmsh node tag.
            # ----------------------------------------------------------

            if a <= nPrimaryLow

                node = Int(nodes[a])

                rnode = vertex_to_reduced[node]

                if rnode == 0

                    nReduced += 1
                    rnode = nReduced

                    vertex_to_reduced[node] = rnode

                    push!(xr, Xlow[a, 1])
                    push!(yr, Xlow[a, 2])
                    push!(zr, Xlow[a, 3])
                end

                rconn[a] = rnode

                continue
            end

            # ----------------------------------------------------------
            # Edge/face/internal reduced node
            # ----------------------------------------------------------

            xa = Xlow[a, 1]
            ya = Xlow[a, 2]
            za = Xlow[a, 3]

            found = 0

            for je in touched

                # Sharing only one primary node means vertex contact.
                neighbour_hits[je] >= 2 || continue

                old_conn = reduced_conn[je]

                for rnode in old_conn

                    dx = xa - xr[rnode]
                    dy = ya - yr[rnode]
                    dz = za - zr[rnode]

                    if dx * dx +
                       dy * dy +
                       dz * dz <= tol2

                        found = rnode
                        break
                    end
                end

                found != 0 && break
            end

            if found == 0

                nReduced += 1
                found = nReduced

                push!(xr, xa)
                push!(yr, ya)
                push!(zr, za)
            end

            rconn[a] = found
        end

        reduced_conn[ie] = rconn

        # --------------------------------------------------------------
        # Reset neighbour workspace
        # --------------------------------------------------------------

        @inbounds for je in touched
            neighbour_hits[je] = 0
        end

        # --------------------------------------------------------------
        # Register current element at its primary nodes
        # --------------------------------------------------------------

        @inbounds for a in 1:nPrimary

            node = Int(nodes[a])

            lst = node_to_elements[node]

            if lst === nothing

                new_list = Int[ie]
                node_to_elements[node] = new_list

            else

                push!(lst, ie)
            end
        end
    end

    # ------------------------------------------------------------------
    # Assemble scalar Tn
    # ------------------------------------------------------------------

    IT = Int[]
    JT = Int[]
    VT = Float64[]

    max_nLow = maximum(nLows)

    sizehint!(IT, P.non * max_nLow)
    sizehint!(JT, P.non * max_nLow)
    sizehint!(VT, P.non * max_nLow)

    full_seen = falses(P.non)

    zero_tol = 100 * eps(Float64)

    @inbounds for ie in 1:nElem

        tid = elem_type_id[ie]

        nodes = elem_nodes[ie]
        rconn = reduced_conn[ie]

        Te = Tes[tid]

        nHigh = nHighs[tid]
        nLow = nLows[tid]

        for a in 1:nHigh

            node = Int(nodes[a])

            full_seen[node] && continue
            full_seen[node] = true

            for b in 1:nLow

                v = Te[a, b]

                abs(v) <= zero_tol && continue

                push!(IT, node)
                push!(JT, rconn[b])
                push!(VT, v)
            end
        end
    end

    Tn =
        sparse(
            IT,
            JT,
            VT,
            P.non,
            nReduced
        )

    # ------------------------------------------------------------------
    # Assemble scalar Rn
    #
    # For a shared reduced node, the interpolation row from the first
    # processed element containing that node is used.
    # ------------------------------------------------------------------

    IR = Int[]
    JR = Int[]
    VR = Float64[]

    max_nHigh = maximum(nHighs)

    sizehint!(IR, nReduced * max_nHigh)
    sizehint!(JR, nReduced * max_nHigh)
    sizehint!(VR, nReduced * max_nHigh)

    reduced_seen = falses(nReduced)

    @inbounds for ie in 1:nElem

        tid = elem_type_id[ie]

        nodes = elem_nodes[ie]
        rconn = reduced_conn[ie]

        Re = Res[tid]

        nHigh = nHighs[tid]
        nLow = nLows[tid]

        for a in 1:nLow

            rnode = rconn[a]

            reduced_seen[rnode] && continue
            reduced_seen[rnode] = true

            for b in 1:nHigh

                v = Re[a, b]

                abs(v) <= zero_tol && continue

                push!(IR, rnode)
                push!(JR, Int(nodes[b]))
                push!(VR, v)
            end
        end
    end

    Rn =
        sparse(
            IR,
            JR,
            VR,
            nReduced,
            P.non
        )

    # ------------------------------------------------------------------
    # Expand according to P.pdim
    # ------------------------------------------------------------------

    d = P.pdim

    if d == 1
        return Tn, Rn
    end

    Id =
        spdiagm(
            0 => ones(Float64, d)
        )

    T = kron(Tn, Id)
    R = kron(Rn, Id)

    return T, R
end

reductionMatrices

In [308]:
structured_rect_mesh(lx=2, order=3, n=50)

In [309]:
material = Material("body")

Pp = Problem([material], type=:ScalarField, field=:p, rhs_field=:fp, dim=2)
Pv = Problem([material], type=:VectorField, field=:v, rhs_field=:fv, dim=2);

In [310]:
using SparseArrays

"""
    q2_to_q1_transformation(P)

Build a sparse prolongation matrix that embeds a Q1 scalar field
into the Q2 nodal space of a quadrilateral mesh.

The returned matrix `T` satisfies

    p_Q2 = T * p_Q1

where Q2 edge nodes are interpolated from the two adjacent corner nodes
and the center node is interpolated from all four corner nodes.
"""
function q2_to_q1_transformation(P::Problem)

    gmsh.model.setCurrent(P.name)

    # --- collect all 2D elements belonging to the problem ---
    relations = Dict{Int,Dict{Int,Float64}}()
    primary_nodes = Set{Int}()

    function set_relation!(node, cols, vals)
        rel = Dict(Int(c) => Float64(v) for (c, v) in zip(cols, vals))

        if haskey(relations, node)
            relations[node] == rel ||
                error("Inconsistent Q2→Q1 relation for node $node.")
        else
            relations[node] = rel
        end
    end

    for mat in P.material
        dimTags = gmsh.model.getEntitiesForPhysicalName(mat.phName)

        for (edim, etag) in dimTags
            edim == 2 || continue

            elemTypes, elemTags, elemNodeTags =
                gmsh.model.mesh.getElements(edim, etag)

            for it in eachindex(elemTypes)

                et = elemTypes[it]

                _, _, order, numNodes, _, numPrimaryNodes =
                    gmsh.model.mesh.getElementProperties(et)

                order == 2 ||
                    error("q2_to_q1_transformation requires second-order elements.")

                numPrimaryNodes == 4 ||
                    error("Only quadrilateral Q2 elements are supported in this prototype.")

                numNodes == 9 ||
                    error("This prototype expects 9-node Q2 quadrilaterals.")

                conn = elemNodeTags[it]
                nel = length(elemTags[it])

                for e in 1:nel
                    o = (e - 1) * numNodes

                    n1 = Int(conn[o+1])
                    n2 = Int(conn[o+2])
                    n3 = Int(conn[o+3])
                    n4 = Int(conn[o+4])

                    n5 = Int(conn[o+5])
                    n6 = Int(conn[o+6])
                    n7 = Int(conn[o+7])
                    n8 = Int(conn[o+8])
                    n9 = Int(conn[o+9])

                    union!(primary_nodes, (n1, n2, n3, n4))

                    # corner nodes
                    set_relation!(n1, [n1], [1.0])
                    set_relation!(n2, [n2], [1.0])
                    set_relation!(n3, [n3], [1.0])
                    set_relation!(n4, [n4], [1.0])

                    # edge nodes
                    set_relation!(n5, [n1, n2], [0.5, 0.5])
                    set_relation!(n6, [n2, n3], [0.5, 0.5])
                    set_relation!(n7, [n3, n4], [0.5, 0.5])
                    set_relation!(n8, [n4, n1], [0.5, 0.5])

                    # center node
                    set_relation!(
                        n9,
                        [n1, n2, n3, n4],
                        [0.25, 0.25, 0.25, 0.25]
                    )
                end
            end
        end
    end

    primary = sort!(collect(primary_nodes))
    reduced_index = Dict(node => i for (i, node) in enumerate(primary))

    I = Int[]
    J = Int[]
    V = Float64[]

    for node in sort!(collect(keys(relations)))
        for (master, weight) in relations[node]
            push!(I, node)
            push!(J, reduced_index[master])
            push!(V, weight)
        end
    end

    T = sparse(I, J, V, P.non, length(primary))

    return T, primary
end

q2_to_q1_transformation

In [311]:
pres1 = BoundaryCondition("rightbottom", problem=Pp, p=0);

In [312]:
suppT = BoundaryCondition("top", problem=Pv, vx=0, vy=0)
suppB = BoundaryCondition("bottom", problem=Pv, vx=0, vy=0);

In [313]:
load_v = LoadCondition("body", fvx=1.0, fvy=0.0)
fv = loadVector(Pv, [load_v])

gp = loadVector(Pp, [])

F = SystemVector([fv, gp]);

In [314]:
μ = 1.0

A = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ)

B = ∫(Div(Pv) ⋅ Pp);

In [315]:
γ = 1e-1          # grad-div ( 1e-2...1e0)
δ = 0             # pressure Laplacian (mesh dependent)

C = ∫(Grad(Pp) ⋅ Grad(Pp) * δ)

D = ∫(Div(Pv) ⋅ Div(Pv) * γ);

In [316]:
# alternative way
@time AD = ∫((SymGrad(Pv) ⋅ SymGrad(Pv)) * 2μ + γ * (Div(Pv) ⋅ Div(Pv)));

  1.498030 seconds (503.83 k allocations: 571.367 MiB, 24.54% gc time)


In [317]:
K = SystemMatrix([A+D B;
    B' -C])

K[:, :]

136353×136353 SparseMatrixCSC{Float64, Int64} with 9017864 stored entries:
⎡⣿⣿⡛⠛⠛⠿⠿⠿⠿⣿⣛⣛⣛⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⢻⡟⠛⠿⠿⣟⣛⡛⠛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠉⠉⠛⠒⠒⠒⠶⠤⠤⠤⢤⣸⠹⣄⠀⠀⠀⠀⠉⠉⠓⠒⠦⠤⣄⎥
⎢⣿⡄⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡀⠹⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⡇⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⠀⠹⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣧⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⠀⠀⠹⣆⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢸⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⠀⠀⠀⠹⣆⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⡇⠀⠀⠀⠀⠹⣆⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢸⢹⠀⠀⠀⠀⠀⠹⣆⠀⠀⠀⠀⠀⎥
⎢⣿⠀⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⢸⢸⠀⠀⠀⠀⠀⠀⠹⣆⠀⠀⠀⠀⎥
⎢⣿⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⢸⢸⠀⠀⠀⠀⠀⠀⠀⠹⣆⠀⠀⠀⎥
⎢⣿⠀⢸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⢸⢸⠀⠀⠀⠀⠀⠀⠀⠀⠹⣆⠀⠀⎥
⎢⣿⠀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⢸⠘⡇⠀⠀⠀⠀⠀⠀⠀⠀⠹⣆⠀⎥
⎢⣿⠀⠀⣇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⣸⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠹⣆⎥
⎢⣿⠶⣖⡚⠒⠲⠶⠶⠶⠶⠶⠶⠶⠶⣖⣒⣒⣒⣒⣒⣒⣒⣒⠒⠒⠚⠛⠀⠃⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⎥
⎢⣿⡄⠀⠙⠳⢦⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠉⠉⠉⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⢧⠀⠀⠀⠀⠈⠙⠳⢦⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠸⡄⠀⠀⠀⠀⠀⠀⠀⠈⠙⠳⢦⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⢧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⠳⢦⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⠳⢦⣄⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⣿⠀⠀⢧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠙⠳⢦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [318]:
#Tp, pressure_nodes = q2_to_q1_transformation(Pp)
@time Tp, Rnew = reductionMatrices(Pp)

  1.493332 seconds (530.26 k allocations: 59.539 MiB, 98.50% compilation time)


(sparse([1, 104, 105, 899, 900, 5756, 5757, 5758, 5759, 5  …  552, 553, 45448, 45449, 45450, 45451, 45448, 45449, 45450, 45451], [1, 1, 1, 1, 1, 1, 1, 1, 1, 2  …  20300, 20300, 20300, 20300, 20300, 20300, 20301, 20301, 20301, 20301], [1.0, 0.22222222222222224, -0.1111111111111111, -0.1111111111111111, 0.22222222222222227, 0.04938271604938273, -0.024691358024691357, 0.012345679012345675, -0.024691358024691357, 1.0  …  0.888888888888889, 0.8888888888888888, -0.09876543209876543, -0.09876543209876544, 0.19753086419753083, 0.1975308641975308, 0.7901234567901234, 0.7901234567901234, 0.7901234567901235, 0.7901234567901234], 45451, 20301), sparse([1, 5, 8, 9, 20100, 20102, 20103, 20105, 20298, 20299  …  20297, 20301, 20297, 20297, 20297, 20297, 20301, 20301, 20301, 20301], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  45443, 45443, 45444, 45445, 45446, 45447, 45448, 45449, 45450, 45451], [1.0000000000000002, -0.06249999999999986, -0.06249999999999989, 0.0039062500000000555, 1.0, -0.06250000000000011, -0.

In [319]:
nv = size(A.A, 1)

Tv = spdiagm(0 => ones(nv))

90902×90902 SparseMatrixCSC{Float64, Int64} with 90902 stored entries:
⎡⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⢄⎦

In [320]:
Zvp = spzeros(size(Tv, 1), size(Tp, 2))
Zpv = spzeros(size(Tp, 1), size(Tv, 2))

T = [
    Tv Zvp
    Zpv Tp
]

136353×111203 SparseMatrixCSC{Float64, Int64} with 336953 stored entries:
⎡⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⠶⠶⣖⣒⡒⠒⠒⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢰⡄⠀⠀⠀⠉⠉⠓⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⡄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⡄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢻⡄⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀

In [321]:
KT = K.A * T
Kr = T' * KT

fr = T' * F.a

111203×1 Matrix{Float64}:
 6.25000000000006e-6
 0.0
 6.250000000000197e-6
 0.0
 6.24999999999919e-6
 0.0
 6.249999999998632e-6
 0.0
 1.2499999999999992e-5
 0.0
 1.250000000000002e-5
 0.0
 1.249999999999977e-5
 ⋮
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [322]:
println("Full system")
println("  size = ", size(K.A))
println("  nnz  = ", nnz(K.A))

println("\nT")
println("  size = ", size(T))
println("  nnz  = ", nnz(T))

println("\nK*T")
println("  size = ", size(KT))
println("  nnz  = ", nnz(KT))

println("\nReduced system")
println("  size = ", size(Kr))
println("  nnz  = ", nnz(Kr))

Full system
  size = (136353, 136353)
  nnz  = 9017864

T
  size = (136353, 111203)
  nnz  = 336953

K*T
  size = (136353, 111203)
  nnz  = 7976732

Reduced system
  size = (111203, 111203)
  nnz  = 6935600


In [323]:
fixed_v = constrainedDoFs(Pv, [suppT, suppB])

1204-element Vector{Int64}:
    6
  906
 1104
 1106
  908
 1108
 1110
  910
 1112
 1114
  912
 1116
 1118
    ⋮
  201
  591
  593
  203
  595
  597
  205
  599
  601
    3
  603
  605

In [324]:
fixed_p_full = constrainedDoFs(Pp, [pres1])

1-element Vector{Int64}:
 2

In [325]:
fixed_p_red = Int[]

for i in fixed_p_full
    _, cols, vals = findnz(Tp[i:i, :])

    length(cols) == 1 ||
        error("Pressure constraint is not located on a Q1 node.")

    vals[1] ≈ 1.0 ||
        error("Unexpected pressure transformation weight.")

    push!(fixed_p_red, cols[1])
end

In [326]:
nv_red = size(Tv, 2)

fixed = sort(unique(vcat(
    fixed_v,
    nv_red .+ fixed_p_red
)))

1205-element Vector{Int64}:
      1
      2
      3
      4
      5
      6
      7
      8
      9
     10
     11
     12
     13
      ⋮
   1492
   1493
   1494
   1495
   1496
   1497
   1498
   1499
   1500
   1501
   1502
 111002

In [327]:
free = setdiff(1:size(Kr, 1), fixed)

xr = zeros(size(Kr, 1))

xr[free] = Kr[free, free] \ fr[free, 1]

109998-element Vector{Float64}:
  0.010469821022036583
 -0.006208103581134157
  0.015995245299030715
 -0.009156084787475934
  0.0204826975512859
 -0.011260138468473267
  0.024484152781052242
 -0.012829944882735632
  0.02816590408640066
 -0.013995832134936909
  0.031600696867130844
 -0.014828295990543168
  0.03482506284904039
  ⋮
 -0.300880891383399
 -0.19993864395205738
 -0.2739860031874079
 -0.23436761609944784
 -0.6092091879163425
 -0.23633774191567258
 -0.430081385480937
 -0.36976684292548556
 -2.914489739639023
 -0.34854906659921264
 -1.0566212934362738
 -0.914565212485837

In [328]:
x = T * xr

136353-element Vector{Float64}:
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  0.0
  ⋮
 -0.23751823054040716
 -0.2588989085601889
 -0.4386189425009584
 -0.4727155389193627
 -0.365871935095016
 -0.30764657772208354
 -0.3632927242842123
 -0.4275897925131114
 -0.8431051584563408
 -0.6534997649993431
 -1.0002417827343653
 -0.9626877691380047

In [329]:
vdata = reshape(x[1:(Pv.non*Pv.pdim)], :, 1)

v = VectorField(
    [],
    vdata,
    [0.0],
    [],
    1,
    :v2D,
    Pv
)

nodal VectorField
[0.0; 0.0; … ; 0.0023108177290428; 0.00032732710877296766;;]

In [330]:
offset_p = Pv.non * Pv.pdim

pdata = reshape(
    x[(offset_p+1):(offset_p+Pp.non)],
    :,
    1
)

p = ScalarField(
    [],
    pdata,
    [0.0],
    [],
    1,
    :scalar,
    Pp
)

nodal ScalarField
[2.917650342285956; 0.0; … ; -1.0002417827343653; -0.9626877691380047;;]

In [331]:
#v, p = solveField(K, F, support=[pres1, suppT, suppB]);

In [332]:
showDoFResults(v, name="v", visible=true)
showDoFResults(p, name="p");

In [333]:
vv = expandTo3D(v)
∫(Pv, "body", (∇⋅vv)^2)

8.588118084782532e-5

In [334]:
openPostProcessor()

In [335]:
Tp, Rnew = reductionMatrices(Pp)

(sparse([1, 104, 105, 899, 900, 5756, 5757, 5758, 5759, 5  …  552, 553, 45448, 45449, 45450, 45451, 45448, 45449, 45450, 45451], [1, 1, 1, 1, 1, 1, 1, 1, 1, 2  …  20300, 20300, 20300, 20300, 20300, 20300, 20301, 20301, 20301, 20301], [1.0, 0.22222222222222224, -0.1111111111111111, -0.1111111111111111, 0.22222222222222227, 0.04938271604938273, -0.024691358024691357, 0.012345679012345675, -0.024691358024691357, 1.0  …  0.888888888888889, 0.8888888888888888, -0.09876543209876543, -0.09876543209876544, 0.19753086419753083, 0.1975308641975308, 0.7901234567901234, 0.7901234567901234, 0.7901234567901235, 0.7901234567901234], 45451, 20301), sparse([1, 5, 8, 9, 20100, 20102, 20103, 20105, 20298, 20299  …  20297, 20301, 20297, 20297, 20297, 20297, 20301, 20301, 20301, 20301], [1, 1, 1, 1, 2, 2, 2, 2, 3, 3  …  45443, 45443, 45444, 45445, 45446, 45447, 45448, 45449, 45450, 45451], [1.0000000000000002, -0.06249999999999986, -0.06249999999999989, 0.0039062500000000555, 1.0, -0.06250000000000011, -0.

In [353]:
@time Tp, Rnew = reductionMatrices(Pp)

println(size(Tp), "  nnz = ", nnz(Tp))
println(size(Rnew), "  nnz = ", nnz(Rnew))

println(norm(Rnew * Tp - I, Inf))

  0.023367 seconds (35.73 k allocations: 37.243 MiB)
(45451, 20301)  nnz = 246051
(20301, 45451)  nnz = 125751
4.440892098500626e-16


In [351]:
@time Kr = T' * K.A * T

  0.387205 seconds (46 allocations: 716.050 MiB, 2.21% gc time)


111203×111203 SparseMatrixCSC{Float64, Int64} with 6935600 stored entries:
⎡⣿⣿⡛⠛⠻⠿⠿⠿⠿⣿⣛⣛⣛⣛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠛⠻⢿⣛⠛⠛⠛⠛⠛⎤
⎢⣿⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠉⠉⠛⠒⠒⠒⠶⠤⠤⠤⣤⣀⣀⣀⣀⠀⠀⠀⠀⠈⠙⠲⢤⣄⡀⎥
⎢⣿⡆⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠉⢹⡀⠀⠀⠀⠀⠈⠉⎥
⎢⣿⡇⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⠀⎥
⎢⣿⣧⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⡀⠀⠀⠀⠀⠀⎥
⎢⣿⢸⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⠀⎥
⎢⣿⢸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⡀⠀⠀⠀⠀⎥
⎢⣿⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⠀⎥
⎢⣿⠀⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⡀⠀⠀⠀⎥
⎢⣿⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣇⠀⠀⠀⎥
⎢⣿⠀⢸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⠀⎥
⎢⣿⠀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣧⠀⠀⎥
⎢⣿⠀⠀⣧⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢹⡄⠀⎥
⎢⣿⠀⠀⢸⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠈⣧⠀⎥
⎢⣿⠀⠀⢸⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⢸⡄⎥
⎢⣿⠀⠀⠀⡇⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⣧⎥
⎢⣿⣆⠀⠀⠓⠲⠦⢤⣄⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⠀⠀⠀⠀⠀⠀⠘⎥
⎢⣿⠘⣆⠀⠀⠀⠀⠀⠀⠈⠉⠙⠓⠲⠦⢤⣄⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⣿⠀⠘⣆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠉⠙⠓⠶⠦⣤⣄⣀⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⣿⠀⠀⠹⡆⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠉⠛⠒⠶⠤⣤⣀⠀⠀⠀⠀⠀⠀⠀⎦

In [358]:
@time KT = K.A * T
println(size(KT), " nnz = ", nnz(KT))

@time Kr2 = T' * KT
println(size(Kr2), " nnz = ", nnz(Kr2))

  0.195262 seconds (21 allocations: 376.831 MiB, 5.00% gc time)
(136353, 111203) nnz = 7976732
  0.215087 seconds (34 allocations: 339.434 MiB, 0.62% gc time)
(111203, 111203) nnz = 6935600
